# Setup

In [1]:
%load_ext autoreload
%autoreload 2

## IMPORTS
import os

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.geometry import Point

import src.data_filepaths as data_filepaths
import src.utils.loaders as loaders
import src.utils.vis.colors_utils as colors_utils
from src.utils.df_tools import map_values_b_to_bp
from src.utils.loaders import process_loaded_building_df
from src.utils.vis.map import CategoricalColorMap, MapExplorer

# configs
sns.set_theme()

# Overture

Manual changes to overture building data, in lieu of the late september update

In [2]:
b_gdf = gpd.read_parquet(data_filepaths.OVERTURE_BUILDING)

In [3]:
b_gdf["subtype"].unique()

<ArrowStringArray>
[             nan,     'industrial',     'commercial', 'transportation',
    'residential',    'outbuilding',          'civic',        'service',
      'education',  'entertainment',        'medical',      'religious',
       'military',   'agricultural']
Length: 14, dtype: str

In [4]:
b_gdf["class"].unique()

<ArrowStringArray>
[                 nan,         'industrial',             'office',
            'parking',          'dormitory',     'transportation',
               'roof',          'warehouse',         'commercial',
        'residential',      'train_station',          'boathouse',
             'retail',            'service',             'garage',
       'storage_tank',        'manufacture',                'hut',
       'fire_station',               'shed',         'university',
            'shelter',             'public',         'grandstand',
            'college',         'apartments',              'house',
             'school',             'church',           'bungalow',
             'temple',             'mosque',             'chapel',
             'bunker',             'hangar',         'government',
            'library',              'civic',       'kindergarten',
     'farm_auxiliary',        'sports_hall',           'pavilion',
           'hospital',            'toilets'

## Changes

For the ones that receive building types, those are typically the ones in some kinda ambiguous/removed zone,
and as such do not have any usable authoritative source for the building's usage

This is mostly just for the August edition of the data
by the September edition, some of them will be fixed (my own OSM changes), 
and I'm not sure these ids will even stay the same

In [5]:
# SIA Airline House
mask = b_gdf["id"] == "73e77c10-c776-4253-be29-85931e215092"
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "office"

# SIA Training Centre
mask = b_gdf["id"] == "8e78d0a0-23d0-48f5-8825-2813305a04cf"
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "office"

# Changi Cargo buildings
mask = b_gdf["id"].isin(
    [
        "d58ba695-86df-4d2d-bf98-53e2e35f7c4a",
        "bc7b8516-a0b4-46aa-9c2d-8ee1f22c7595",
        "7b6832ba-7903-404e-a41b-65c24945abb0",
        "5fa79839-e7eb-48dc-87b0-22cad630ed2a",
    ]
)
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "warehouse"

# CMPB Building
mask = b_gdf["id"] == "d75ffb73-4a82-46d6-804d-b72bd2fd8687"
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "commercial"

# DSTA Tower A
mask = b_gdf["id"] == "1a727f62-2138-44e3-898c-7763b8020bbd"
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "office"

# Harbourfront towers, because SDP
mask = b_gdf["id"].isin(
    [
        "f8abd9a2-235e-45c9-ab8e-b2d864129f92",
        "59282f59-dc93-40a1-af40-4aa32d289daf",
        "a2de212f-96be-413f-bf25-d72bc1c8e5ce",
    ]
)
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "office"

# Asia Square Towers - lower section
mask = b_gdf["id"] == "978cd6c2-1405-4e7f-8fa2-1e1e74501bfe"
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "commercial"

# SCAPE - though commercial is a weird tagging
mask = b_gdf["id"] == "d15b0d55-ad3f-4955-86ed-ac84ee3e53cb"
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "commercial"

# warehouses at Paya Lebar AB
mask = b_gdf["id"].isin(
    ["e665aed9-a358-44b0-a119-83643308c2ce", "e2c9e37e-dc19-42cd-9ef1-ee4b9c629b49"]
)
b_gdf.loc[mask, "subtype"] = "industrial"
b_gdf.loc[mask, "class"] = "warehouse"

# airport industrial buildings west of Changi
mask = b_gdf["id"].isin(
    [
        "df01edba-9e5a-4c43-9ae1-c5a8f7d8b53e",
        "8ade1ef1-5620-4500-ba99-e43cec6f24a2",
        "8fe42e7a-d3c0-493b-83d2-e0cdefde330c",
    ]
)
b_gdf.loc[mask, "subtype"] = "industrial"
b_gdf.loc[mask, "class"] = "industrial"

# cleantech three
mask = b_gdf["id"] == "cbbf9c6f-04e1-4c8b-a9bd-37feaaa8c159"
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "office"
b_gdf.loc[mask, "num_floors"] = 6

# surbana Jurong Campus - secondary buildings
mask = b_gdf["id"].isin(
    [
        "4e8227a9-dbf9-45e4-a67a-675341d853e1",
        "2d9c6be8-037c-43e4-a724-5f71fc4ed1d3",
        "10de2f7d-ca28-454a-820b-d5c0cb05c318",
    ]
)
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "office"

# PSA Offices TJPG, PSA Horizons
mask = b_gdf["id"].isin(
    ["786c8e7d-5022-4842-a35e-8bbe9706518e", "98451866-eb57-4ef1-995a-9aa409d5dea7"]
)
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "office"

# TJPG Distripark
mask = b_gdf["id"].isin(
    ["5ce506ac-2b9b-4e72-8000-e29a9e21c562", "238ce49f-e9b0-4229-afc5-93ff4a43a890"]
)
b_gdf.loc[mask, "subtype"] = "industrial"
b_gdf.loc[mask, "class"] = "warehouse"


## ERRONEOUS MP ZONES (typically B2/B1 -> industry)

# Tai Seng Exchange, force as office
mask = b_gdf["id"] == "8dd8ba69-08f4-4020-8c3b-4a14302c9676"
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "office"

# IMM
mask = b_gdf["id"] == "8baa3c70-3540-436e-9214-829b85aa8d3a"
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "commercial"

### RESIDENTIAL / DOWNTOWN TWEAKS
# Marina Bay Residences
mask = b_gdf["id"] == "b9bf1e7c-9fe0-4cec-80f3-d35ec4451702"
b_gdf.loc[mask, "subtype"] = "residential"
b_gdf.loc[mask, "class"] = "residential"
# MBS Hotel Towers
mask = b_gdf["id"].isin(
    [
        "719402a8-c136-451d-82ed-31d54990e199",
        "72e4fc04-b68f-4e00-9244-740c26e939f8",
        "7d232bb7-aed7-4f8c-b4a8-f26a0fc0366c",
    ]
)
b_gdf.loc[mask, "subtype"] = "commercial"
b_gdf.loc[mask, "class"] = "hotel"

In [6]:
b_gdf.loc[mask]

,id,names,sources,level,height,min_height,is_underground,num_floors,num_floors_underground,min_floor,...,roof_direction,roof_orientation,roof_color,roof_height,geometry,has_parts,version,bbox,theme,type
102945,7d232bb7-aed7-4f8c-b4a8-f26a0fc0366c,"{'primary': 'Marina Bay Sands Tower 1', 'commo...","[{'property': '', 'dataset': 'OpenStreetMap', ...",NaN,193.0,NaN,False,55.0,NaN,NaN,...,NaN,NaN,NaN,NaN,"POLYGON ((103.86023 1.28232, 103.86053 1.28275...",False,1,"{'xmin': 103.8598403930664, 'xmax': 103.860527...",buildings,building
102959,72e4fc04-b68f-4e00-9244-740c26e939f8,"{'primary': 'Marina Bay Sands Tower 2', 'commo...","[{'property': '', 'dataset': 'OpenStreetMap', ...",NaN,193.0,NaN,False,55.0,NaN,NaN,...,NaN,NaN,NaN,NaN,"POLYGON ((103.86089 1.2838, 103.8606 1.28393, ...",False,1,"{'xmin': 103.86036682128906, 'xmax': 103.86101...",buildings,building
102961,719402a8-c136-451d-82ed-31d54990e199,"{'primary': 'Marina Bay Sands Tower 3', 'commo...","[{'property': '', 'dataset': 'OpenStreetMap', ...",NaN,193.0,NaN,False,55.0,NaN,NaN,...,NaN,NaN,NaN,NaN,"POLYGON ((103.86104 1.28422, 103.86118 1.28469...",False,1,"{'xmin': 103.86077117919922, 'xmax': 103.86117...",buildings,building


## Output

In [7]:
b_gdf.to_parquet(data_filepaths.OVERTURE_BUILDING)